# Practical 02: Data Hygiene: Missing Value Imputation, Deduplication & Renaming

**UNIT 1: Introduction to Big Data and Large Dataset Handling**

**Problem Statement:** Write a Python program to perform data hygiene on a dataset with missing values and duplicate rows. Identify missing entries using isna().sum(), demonstrate both dropping rows (dropna()) and imputing missing numerical values with fillna(mean/median), remove duplicate records using drop_duplicates(), and rename columns using rename().

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None, "display.width", 200)

rnd = np.random.default_rng(123)
n = 190
tbl = pd.DataFrame({
    "Order ID": np.arange(1001, 1001 + n),
    "Customer Name": [f"User_{i:03d}" for i in rnd.integers(1, 100, n)],
    "City": rnd.choice(["Surat", "Mumbai", "Pune", "Nagpur", "Thane", "Rajkot"], n),
    "Product Category": rnd.choice(["Electronics", "Clothing", "Groceries", "Furniture", "Beauty"], n),
    "Quantity": rnd.integers(1, 9, n).astype(float),
    "Unit Price": np.round(rnd.uniform(100, 4000, n), 2),
    "Discount %": rnd.choice([0, 5, 10, 15, 20], n).astype(float),
    "Order Date": pd.date_range("2025-01-20", periods=n, freq="D").strftime("%Y-%m-%d"),
})
for col, k in [("Quantity", 13), ("Unit Price", 8), ("Discount %", 12), ("City", 10)]:
    tbl.loc[rnd.choice(n, k, replace=False), col] = np.nan
tbl = pd.concat([tbl, tbl.sample(11, random_state=5)])
tbl = tbl.sample(frac=1, random_state=9).reset_index(drop=True)

print("Original shape:", tbl.shape)
print(tbl.head())

print(tbl.isna().sum())

df_removed = tbl.dropna()
print(f"dropna(): {tbl.shape[0]} -> {df_removed.shape[0]} rows ({tbl.shape[0] - df_removed.shape[0]} rows lost)")

df_fill = tbl.copy()
df_fill["Quantity"] = df_fill["Quantity"].fillna(df_fill["Quantity"].median())
df_fill["Unit Price"] = df_fill["Unit Price"].fillna(df_fill["Unit Price"].mean())
df_fill["Discount %"] = df_fill["Discount %"].fillna(df_fill["Discount %"].mean())
df_fill["City"] = df_fill["City"].fillna(df_fill["City"].mode()[0])

print("Missing values after imputation:")
print(df_fill.isna().sum())

print("Duplicate rows found:", df_fill.duplicated().sum())
df_out = df_fill.drop_duplicates().reset_index(drop=True)
print("Shape after drop_duplicates():", df_out.shape)

df_out = df_out.rename(columns={
    "Order ID": "oid",
    "Customer Name": "cust_name",
    "City": "town",
    "Product Category": "prod_cat",
    "Quantity": "qty",
    "Unit Price": "unit_cost",
    "Discount %": "disc_percent",
    "Order Date": "ordered_on",
})
print(df_out.columns.tolist())
print(df_out.head())

Original shape: (201, 8)

,Order ID,Customer Name,City,Product Category,Quantity,Unit Price,Discount %,Order Date
0,1085,User_091,Surat,Electronics,1.0,725.37,15.0,2025-04-14
1,1061,User_052,Nagpur,Beauty,8.0,3481.41,10.0,2025-03-21
2,1055,User_008,Nagpur,Clothing,6.0,2136.01,0.0,2025-03-15
3,1110,User_058,Mumbai,Electronics,NaN,3744.74,NaN,2025-05-09
4,1157,User_010,Mumbai,Electronics,4.0,2513.64,5.0,2025-06-25


Order ID             0
Customer Name        0
City                10
Product Category     0
Quantity            14
Unit Price           9
Discount %          13
Order Date           0
dtype: int64
dropna(): 201 -> 158 rows (43 rows lost)
Missing values after imputation:
Order ID            0
Customer Name       0
City                0
Product Category    0
Quantity            0
Unit Price          0
Discount %          0
Order Date          0
dtype: int64
Duplicate rows found: 11
Shape after drop_duplicates(): (190, 8)
['oid', 'cust_name', 'town', 'prod_cat', 'qty', 'unit_cost', 'disc_percent', 'ordered_on']

,oid,cust_name,town,prod_cat,qty,unit_cost,disc_percent,ordered_on
0,1085,User_091,Surat,Electronics,1.0,725.37,15.000000,2025-04-14
1,1061,User_052,Nagpur,Beauty,8.0,3481.41,10.000000,2025-03-21
2,1055,User_008,Nagpur,Clothing,6.0,2136.01,0.000000,2025-03-15
3,1110,User_058,Mumbai,Electronics,4.0,3744.74,9.441489,2025-05-09
4,1157,User_010,Mumbai,Electronics,4.0,2513.64,5.000000,2025-06-25
